## Airbnb Chatbot
Collaborators: Ben Ghouzi, Shawn Lokshin, Khadiatou Ly, Veronica Song

Goal: Build a chatbot that can process user's natural language queries to recommend experiences and services based on user-user filtering and selected housing location

In [1]:
import numpy as np
import pandas as pd

listings = pd.read_csv("data/listings.csv")
reviews = pd.read_csv("data/reviews.csv")
user_booking = pd.read_csv("data/user_bookings.csv")

user_booking.head()

,user_id,listing_id,city,trip_purpose,guests,nights_stayed,total_paid,rating,booking_month
0,U0145,L0290,Chicago,group,4,3,834,5,10
1,U0189,L0337,New York,leisure,2,3,768,4,7
2,U0172,L0016,Boston,leisure,2,7,1197,3,3
3,U0020,L0465,Miami,romantic,2,7,826,5,2
4,U0024,L0148,Boston,leisure,1,5,550,5,12


## Sentiment Analysis

Assigns a sentiment score to each user review

In [3]:
# !python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 13.4 MB/s  0:00:00 eta 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [4]:
# using clustering text analysis to determine sentiment from user reviews
# goal: assign a numeric rating to each users review from reviews.csv

# Install dependencies if needed:
# pip install pandas spacy nltk vaderSentiment

import spacy
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import nltk

# Download punkt tokenizer if not already
nltk.download('punkt')

# Load spaCy English model
# python -m spacy download en_core_web_sm
nlp = spacy.load("en_core_web_sm")

# Initialize sentiment analyzer
analyzer = SentimentIntensityAnalyzer()

# -------------------------------
# 1. Load Data
# -------------------------------

# Ensure column exists
assert "review_text" in reviews.columns

# -------------------------------
# 2. Phrase Extraction Function
# -------------------------------
def extract_phrases(text):
    """
    Extract meaningful phrases using noun chunks + sentence splits
    """
    doc = nlp(str(text))
    
    phrases = []
    
    # Sentence-level splitting
    for sent in doc.sents:
        phrases.append(sent.text.strip())
    
    # Add noun chunks for finer granularity
    for chunk in doc.noun_chunks:
        phrases.append(chunk.text.strip())
    
    return list(set(phrases))  # remove duplicates

# -------------------------------
# 3. Phrase Sentiment Scoring
# -------------------------------
def score_phrase(phrase):
    """
    Returns compound sentiment score (-1 to +1)
    """
    score = analyzer.polarity_scores(phrase)
    return score['compound']

# -------------------------------
# 4. Review-Level Aggregation
# -------------------------------
def compute_review_sentiment(text):
    phrases = extract_phrases(text)
    
    if not phrases:
        return 0, []
    
    phrase_scores = []
    
    for phrase in phrases:
        score = score_phrase(phrase)
        phrase_scores.append((phrase, score))
    
    # Weighted aggregation (longer phrases = higher weight)
    weighted_sum = 0
    total_weight = 0
    
    for phrase, score in phrase_scores:
        weight = len(phrase.split())
        weighted_sum += score * weight
        total_weight += weight
    
    overall_score = weighted_sum / total_weight if total_weight != 0 else 0
    
    return overall_score, phrase_scores

# -------------------------------
# 5. Apply to Dataset
# -------------------------------
results = reviews["review_text"].apply(compute_review_sentiment)

reviews["sentiment_score"] = results.apply(lambda x: x[0])
reviews["phrase_sentiments"] = results.apply(lambda x: x[1])

# -------------------------------
# 6. Optional: Label Categories
# -------------------------------
def label_sentiment(score):
    if score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    else:
        return "neutral"

reviews["sentiment_label"] = reviews["sentiment_score"].apply(label_sentiment)

# -------------------------------
# 7. Save Results
# -------------------------------
reviews.to_csv("reviews_with_sentiment.csv", index=False)

# -------------------------------
# 8. Preview Example
# -------------------------------
print(reviews[["review_text", "sentiment_score", "sentiment_label"]].head())

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/veronicasong/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


                                         review_text  sentiment_score  \
0  Stayed here for a week and loved every minute....         0.336859   
1  Great location, just 9 minutes from Lexington ...         0.368300   
2  Beautiful apartment in a great part of New Yor...         0.354510   
3  Perfect for our business trip. Close to Centra...         0.329196   
4  Wonderful host, beautiful space in Midtown Man...         0.369892   

  sentiment_label  
0        positive  
1        positive  
2        positive  
3        positive  
4        positive  


LLM prompt used: Act like a professional data scientist. I am currently trying to build a sentiment text analysis model based off user review text stored in reviews.csv under column review_text. Please code in python a phrase-level sentiment analyser that separates out the user reviews into phrases and then assigns an overall user sentiment score to their review.

# User-user Collaborative Filtering

In [ ]:
# setting up user-user collaborative filtering


LLM prompt used: Act like a professional data scientist. I am currently trying to build a user-user collaborative filtering function that 

## Airbnb Chatbot 

Aggregates all the databases together, takes in basic user queries/ questions and generates a recommendation on a place to stay and all relevant details

In [6]:
# %pip install faiss-cpu sentence-transformers transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 11.5 MB/s  0:00:00eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [7]:
# load and aggregate all the data
reviews = pd.read_csv("reviews_with_sentiment.csv")

# -------------------------------
# Aggregate review insights
# -------------------------------
review_agg = reviews.groupby("listing_id").agg({
    "sentiment_score": "mean",
    "sentiment_label": lambda x: x.value_counts().idxmax()
}).reset_index()

# -------------------------------
# Aggregate booking insights
# -------------------------------
booking_agg = user_booking.groupby("listing_id").agg({
    "rating": "mean",
    "trip_purpose": lambda x: x.value_counts().idxmax()
}).reset_index()

# -------------------------------
# Merge everything
# -------------------------------
df = listings.merge(review_agg, on="listing_id", how="left")
df = df.merge(booking_agg, on="listing_id", how="left")

In [8]:
def create_document(row):
    return f"""
    Listing in {row['city']} ({row['neighborhood']})
    Property type: {row['property_type']}
    Bedrooms: {row['bedrooms']}, Bathrooms: {row['bathrooms']}
    Max guests: {row['max_guests']}
    Price per night: ${row['price_per_night']}
    
    Amenities: {row['amenities']}
    Superhost: {row['superhost']}
    Host rating: {row['host_rating']}
    
    Transit: {row['nearest_transit']} ({row['transit_walk_min']} min walk)
    
    Nearby attractions: {row['nearby_attraction_1']}, {row['nearby_attraction_2']}
    
    Review sentiment: {row.get('sentiment_label', 'unknown')} 
    Avg sentiment score: {row.get('sentiment_score', 0)}
    
    Avg user rating: {row.get('rating', 'N/A')}
    Common trip purpose: {row.get('trip_purpose', 'N/A')}
    """

documents = df.apply(create_document, axis=1).tolist()

In [9]:
# embedding and storing the vectors
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Load embedding model
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Encode documents
doc_embeddings = embedder.encode(documents, show_progress_bar=True)

# Create FAISS index
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(doc_embeddings))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6861.06it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 19/19 [00:03<00:00,  5.79it/s]


In [10]:
# Retrieval function
def retrieve(query, top_k=5):
    query_embedding = embedder.encode([query])
    distances, indices = index.search(query_embedding, top_k)
    
    results = [documents[i] for i in indices[0]]
    return results

In [11]:
# loading the chatbot model
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:05<00:00, 57.07it/s]


In [12]:
# RAG prompt and generation
def generate_answer(query, retrieved_docs):
    context = "\n\n".join(retrieved_docs)
    
    prompt = f"""
    You are a travel assistant.

    User query:
    {query}

    Relevant listings:
    {context}

    Instructions:
    - Recommend the best listing(s)
    - Justify using price, sentiment, ratings, and amenities
    - Mention nearby attractions
    - Keep answer concise but helpful
    """

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.7,
        do_sample=True
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [13]:
# Chat function
def chat(query):
    retrieved_docs = retrieve(query, top_k=5)
    answer = generate_answer(query, retrieved_docs)
    return answer

In [14]:
# testing chatbot with a basic user query
query = "I am planning a business trip in New York. What is the best place to stay and attractions nearby?"

response = chat(query)
print(response)


    You are a travel assistant.

    User query:
    I am planning a business trip in New York. What is the best place to stay and attractions nearby?

    Relevant listings:
    
    Listing in New York (Lower East Side)
    Property type: Shared room
    Bedrooms: 1, Bathrooms: 1.0
    Max guests: 2
    Price per night: $100

    Amenities: Patio; BBQ grill; Pet friendly; Kitchen; Iron; Carbon monoxide alarm; EV charger; Hot tub; Heating; Gym access; Coffee maker
    Superhost: 0
    Host rating: 4.56

    Transit: Canal St (12 min walk)

    Nearby attractions: Broadway Theater District, Central Park

    Review sentiment: positive 
    Avg sentiment score: 0.23689937704248365

    Avg user rating: 4.142857142857143
    Common trip purpose: business
    


    Listing in New York (Upper West Side)
    Property type: Private room
    Bedrooms: 1, Bathrooms: 1.0
    Max guests: 1
    Price per night: $115

    Amenities: BBQ grill; Hot tub; WiFi; Workspace; Pool; Gym access; EV charg

LLM prompt used: Act like a professional software engineer. I am currently trying to build out a basic RAG chatbot that takes in files listings.csv, reviews_with_sentiment.csv, and user_bookings.csv. Listings.csv has information on "listing_id,city,neighborhood,property_type,bedrooms,bathrooms,max_guests,price_per_night,amenities,host_rating,superhost,instant_book,min_nights,nearest_transit,transit_walk_min,nearby_attraction_1,nearby_attraction_2". Reviews_with_sentiment.csv has information on "review_id,listing_id,user_id,review_text,sentiment_score,phrase_sentiments,sentiment_label". User_bookings.csv has information on "user_id,listing_id,city,trip_purpose,guests,nights_stayed,total_paid,rating,booking_month". I want to build a basic RAG chatbot using this basic chatbot model "model_name = "Qwen/Qwen2.5-1.5B-Instruct" that is able to take in a basic user query like "I am planning a business trip in ___ city. What is the best place to stay and the attractions nearby". Please build out a basic chatbot that is able to connect all of these csv files and answer user queries